In [1]:
import os
import random
import numpy as np
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
import imageio.v2 as imageio
import networkx as nx
from torch_geometric.utils import to_networkx, softmax

from torch_geometric.datasets import QM9
from torch_geometric.loader import DataLoader

from torch import nn
from torch_geometric.nn.models.dimenet import DimeNetPlusPlus, triplets, radius_graph
from torch_scatter import scatter
from torch_geometric.nn import MessagePassing
import scipy.sparse as sp
from scipy.sparse.csgraph import connected_components

In [2]:
# Configuration and Hyperparameters
SEED = 50
MIN_NODES = 10
MAX_NODES = 10 
DATA_ROOT = os.environ.get("QM9_ROOT", "~/TPNN/QM9/dataset")
# SAVE_DIR = 'mincut_viz'

# Set seeds for reproducibility
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed(SEED)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
# os.makedirs(SAVE_DIR, exist_ok=True)


BATCH_SIZE = 2
# Load and Preprocess Data
dataset = QM9(root=DATA_ROOT)

# Filter dataset for graphs with > MIN_NODES nodes
filtered_data = [d for d in dataset if d.num_nodes <= MAX_NODES and d.num_nodes >= MIN_NODES]
print(f"Total QM9 molecules: {len(dataset)}")
print(f"Filtered dataset size (< {MAX_NODES} nodes and > {MIN_NODES} nodes): {len(filtered_data)}")

# Split Data
n = len(filtered_data)
train_size = int(0.4 * n)
val_size = int(0.2 * n)
test_size = n - train_size - val_size

train_dataset = filtered_data[:train_size]
val_dataset = filtered_data[train_size:train_size+val_size]
test_dataset = filtered_data[train_size+val_size:]

# Create Loaders
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=1, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=True)

print(f"Train/Val/Test sizes: {len(train_dataset)}/{len(val_dataset)}/{len(test_dataset)}")



Total QM9 molecules: 130831
Filtered dataset size (< 10 nodes and > 10 nodes): 483
Train/Val/Test sizes: 193/96/194


In [3]:
class DimeNetPlusPlusNode(DimeNetPlusPlus):
    """DimeNet++ wrapper that returns per-node embeddings (hidden_channels dim)
    instead of graph-level scalar predictions.

    Pretrained loading (requires TensorFlow):
        model, (train, val, test) = DimeNetPlusPlusNode.from_qm9_pretrained(root, dataset, target)
        # This works because the inherited @classmethod uses cls(...) internally.

    TF-free loading from a saved state dict:
        model = DimeNetPlusPlusNode.from_state_dict(path, **kwargs)
    """

    def forward(self, z, pos, batch=None):
        """
        Forward pass returning node embeddings after all interaction blocks.

        Args:
            z (Tensor): Atomic number of each atom [num_atoms].
            pos (Tensor): Coordinates of each atom [num_atoms, 3].
            batch (LongTensor, optional): Batch vector [num_atoms].

        Returns:
            Tensor: Node embeddings [num_atoms, hidden_channels].
        """
        edge_index = radius_graph(pos, r=self.cutoff, batch=batch,
                                  max_num_neighbors=self.max_num_neighbors)

        i, j, idx_i, idx_j, idx_k, idx_kj, idx_ji = triplets(
            edge_index, num_nodes=z.size(0))

        dist = (pos[i] - pos[j]).pow(2).sum(dim=-1).sqrt()

        # Calculate angles (DimeNet++ formulation)
        pos_jk, pos_ij = pos[idx_j] - pos[idx_k], pos[idx_i] - pos[idx_j]
        a = (pos_ij * pos_jk).sum(dim=-1)
        b = torch.cross(pos_ij, pos_jk, dim=1).norm(dim=-1)
        angle = torch.atan2(b, a)

        rbf = self.rbf(dist)
        sbf = self.sbf(dist, angle, idx_kj)

        # Embedding block — produces edge-level features [E, hidden_channels]
        x = self.emb(z, rbf, i, j)

        # Interaction blocks — x stays edge-level [E, hidden_channels]
        for interaction_block in self.interaction_blocks:
            x = interaction_block(x, rbf, sbf, idx_kj, idx_ji)

        # Scatter edge features to nodes: [E, hidden] -> [N, hidden]
        node_embeddings = scatter(x, i, dim=0, dim_size=pos.size(0), reduce='mean')

        return node_embeddings  # [num_atoms, hidden_channels]

    @classmethod
    def from_state_dict(cls, state_dict_path, **kwargs):
        """Load from a saved PyTorch state dict (no TensorFlow needed).

        Example:
            # First, save a pretrained model's weights:
            #   torch.save(pretrained_model.state_dict(), 'dimenet_pp.pt')
            # Then load into a node-embedding variant:
            #   model = DimeNetPlusPlusNode.from_state_dict('dimenet_pp.pt',
            #       hidden_channels=128, out_channels=1, num_blocks=4,
            #       int_emb_size=64, basis_emb_size=8, out_emb_channels=256,
            #       num_spherical=7, num_radial=6)
        """
        model = cls(**kwargs)
        state_dict = torch.load(state_dict_path, map_location='cpu', weights_only=True)
        model.load_state_dict(state_dict, strict=False)
        return model

In [ ]:
class ScalarKernel(torch.nn.Module):
    def __init__(self, in_channels, hidden_channels, out_channels=1, alpha=10.0):
        super(ScalarKernel, self).__init__()
        self.kernel = torch.nn.Sequential(
            torch.nn.Linear(in_channels, hidden_channels),
            torch.nn.SiLU(),
            torch.nn.Linear(hidden_channels, out_channels),
            torch.nn.Sigmoid()
        )
        self.alpha = alpha

    def smooth_max(self, x):
        return torch.sum(x * torch.exp(self.alpha * x)) / torch.sum(torch.exp(self.alpha * x))
    
    def forward(self, x, edge_index):
        
        node_scalars = self.kernel(x)
        src, dst = edge_index
        
        # Gather values for source and destination nodes
        src_values = node_scalars[src]
        dst_values = node_scalars[dst]
        
        # The edge value is the maximum of its endpoints
        edge_scalars = torch.maximum(src_values, dst_values)
        
        return node_scalars, edge_scalars

def build_hierarchical_graph(node_features, edge_index, batch, node_values, edge_values, grouping_param=0.000):
    """
    Constructs a hierarchical graph structure based on edge values.
    
    Args:
        node_features (Tensor): [N, F]
        edge_index (Tensor): [2, E]
        batch (Tensor): [N]
        node_values (Tensor): [N, 1]
        edge_values (Tensor): [E, 1] OR [E]
        grouping_param (float): Grouping threshold parameter.
    
    Returns:
        combined_features (Tensor): [N_total, F]
        combined_edge_index (Tensor): [2, E_total]
        node_level_ids (Tensor): [N_total]
        edge_level_ids (Tensor): [E_total]
    """
    device = node_features.device
    num_nodes = node_features.shape[0]
    num_features = node_features.shape[1]
    
    # Flatten edge_values if needed
    if edge_values.dim() > 1:
        edge_values = edge_values.squeeze()

    # --- 1. Edge Sorting ---
    sorted_idx = torch.argsort(edge_values)
    sorted_values = edge_values[sorted_idx]
    sorted_u = edge_index[0, sorted_idx]
    sorted_v = edge_index[1, sorted_idx]

    # --- 2. Initialization ---
    all_features = [node_features]
    all_edges_list = [] # Will hold tuples of (src, dst) for inter-layer
    all_level_ids = [torch.zeros(num_nodes, dtype=torch.long, device=device)]
    all_edge_level_ids = []
    all_node_values = [node_values]
    
    
    # Maintains mapping from Original Node ID -> Current Cluster ID
    # Initially, each node is its own cluster (0..N-1)
    # The 'Current Cluster ID' corresponds to the node index in the current top layer.
    global_cluster_map = torch.arange(num_nodes, device=device)
    current_num_clusters = num_nodes
    current_node_values = node_values
    
    current_level = 0
    
    # Determine number of graphs (assuming batch is 0..B-1)
    num_graphs = batch.max().item() + 1
    
    while current_num_clusters > 1:
        
        # --- 3. Identify Candidates ---
        # Map sorted edges to current cluster IDs
        c_u = global_cluster_map[sorted_u]
        c_v = global_cluster_map[sorted_v]
        
        # Filter: Only edges connecting DIFFERENT clusters are valid
        diff_mask = c_u != c_v
        
        if not diff_mask.any():
            break # No more merges possible (e.g., graphs fully connected internally)
            
        # Get active edges
        valid_indices = torch.nonzero(diff_mask).squeeze()
        if valid_indices.dim() == 0:
            valid_indices = valid_indices.unsqueeze(0)
            
        # The lowest value edge connecting different clusters
        first_valid_idx = valid_indices[0]
        min_cluster_diff_val = sorted_values[first_valid_idx]
        
        # Define Threshold
        threshold = min_cluster_diff_val + grouping_param
        
        # Slice the candidate window
        limit_idx = torch.searchsorted(sorted_values, threshold, right=True)
        
        # Filter active edges within the threshold
        cand_indices_mask = valid_indices < limit_idx
        merging_indices = valid_indices[cand_indices_mask]
        
        if merging_indices.numel() == 0:
            break
            
        # Get the clusters that need to merge
        merge_u = c_u[merging_indices]
        merge_v = c_v[merging_indices]
        
        # --- 4. Compute Connected Components (Next Layer) ---
        new_parent_map = torch.arange(current_num_clusters, device=device)
        
        # Iterative Min-Propagate for CC
        for _ in range(40):
            p_u = new_parent_map[merge_u]
            p_v = new_parent_map[merge_v]
            min_p = torch.minimum(p_u, p_v)
            
            try:
                # Optimized scatter reduce for PyTorch 1.12+
                new_parent_map.scatter_reduce_(0, merge_u, min_p, reduce='amin', include_self=True)
                new_parent_map.scatter_reduce_(0, merge_v, min_p, reduce='amin', include_self=True)
            except AttributeError:
                # Fallback loops
                safe_mask_u = new_parent_map[merge_u] > min_p
                if safe_mask_u.any():
                    new_parent_map[merge_u[safe_mask_u]] = min_p[safe_mask_u]
                
                safe_mask_v = new_parent_map[merge_v] > min_p
                if safe_mask_v.any():
                    new_parent_map[merge_v[safe_mask_v]] = min_p[safe_mask_v]

        # Compact the IDs to 0..K-1
        unique_parents, inverse_indices = torch.unique(new_parent_map, return_inverse=True)
        num_new_clusters = unique_parents.size(0)
        
        # inverse_indices maps current cluster IDs to their new parent cluster ID.
        new_layer_node_values = scatter(
            current_node_values, 
            inverse_indices, 
            dim=0, 
            dim_size=num_new_clusters, 
            reduce='mean'
        )
        all_node_values.append(new_layer_node_values)
        current_node_values = new_layer_node_values # Update state for next layer
        
        # --- 5. Build Next Layer ---
        
        # Features: Initialize with Zeros
        new_layer_features = torch.zeros((num_new_clusters, num_features), device=device)
        all_features.append(new_layer_features)
        
        # Level IDs for Nodes
        current_level += 1
        all_level_ids.append(torch.full((num_new_clusters,), current_level, dtype=torch.long, device=device))
        
        # Inter-Level Edges (Bidirectional)
        src_nodes = torch.arange(current_num_clusters, device=device)
        dst_nodes = inverse_indices
        
        all_edges_list.append((src_nodes, dst_nodes))
        
        # Level IDs for Edges (Using lower level index, so level 0 means L0-L1 edge)
        # 2 directions per edge -> 2 * src_nodes.size(0)
        edge_level = current_level - 1
        all_edge_level_ids.append(
            torch.full((src_nodes.size(0) * 2,), edge_level, dtype=torch.long, device=device)
        )
        
        # Update State
        global_cluster_map = inverse_indices[global_cluster_map]
        current_num_clusters = num_new_clusters
        
        if current_num_clusters <= num_graphs:
            break

    # --- 6. Assemble Final Graph ---
    
    # Calculate Node Offsets
    layer_sizes = [f.shape[0] for f in all_features]
    cumulative_sizes = torch.tensor([0] + layer_sizes[:-1], device=device).cumsum(0)
    
    # Concatenate Features
    combined_features = torch.cat(all_features, dim=0)
    node_level_ids = torch.cat(all_level_ids, dim=0)
    
    # Concatenate our tracked continuous values
    combined_node_values = torch.cat(all_node_values, dim=0)
    
    # Construct Edge Index
    final_edge_sources = []
    final_edge_targets = []
    
    for i, (src_local, dst_local) in enumerate(all_edges_list):
        offset_src = cumulative_sizes[i]
        offset_dst = cumulative_sizes[i+1]
        
        u = src_local + offset_src
        v = dst_local + offset_dst
        
        final_edge_sources.append(u)
        final_edge_targets.append(v)
        final_edge_sources.append(v) # Bidirectional
        final_edge_targets.append(u)
        
    if len(final_edge_sources) > 0:
        combined_edge_index = torch.stack([
            torch.cat(final_edge_sources),
            torch.cat(final_edge_targets)
        ], dim=0)
        edge_level_ids = torch.cat(all_edge_level_ids, dim=0)
    else:
        combined_edge_index = torch.empty((2, 0), dtype=torch.long, device=device)
        edge_level_ids = torch.empty((0,), dtype=torch.long, device=device)
    
    return combined_features, combined_edge_index, node_level_ids, edge_level_ids, combined_node_values


class AgnosticUpwardModule(MessagePassing):
    def __init__(self, feature_dim):
        super(AgnosticUpwardModule, self).__init__(aggr='add') 
        self.mlp = nn.Sequential(
            nn.Linear(feature_dim * 2, feature_dim),
            nn.SiLU(),
            nn.Linear(feature_dim, feature_dim),
            nn.SiLU(),
            nn.Linear(feature_dim, feature_dim)
        )

    def forward(self, x, edge_index, node_values):
        # We pass node_values explicitly into the propagation step
        return self.propagate(edge_index, x=x, node_values=node_values)

    def message(self, x_i, x_j, node_values_i, node_values_j, index, ptr, size_i):
        tmp = torch.cat([x_i, x_j], dim=1)
        msg = self.mlp(tmp)
        
        # 1. Calculate the raw, unnormalized score (closer values = score closer to 0, further = more negative)
        raw_score = -torch.abs(node_values_i - node_values_j)
        
        # 2. Apply Softmax grouped by the destination node (index).
        # This guarantees that for any target node, the sum of incoming edge_weights == 1.0
        edge_weight = softmax(raw_score, index, ptr, size_i)
        return msg * edge_weight


class AgnosticDownwardModule(MessagePassing):
    def __init__(self, feature_dim):
        super(AgnosticDownwardModule, self).__init__(aggr='add')
        self.mlp = nn.Sequential(
            nn.Linear(feature_dim * 2, feature_dim),
            nn.SiLU(),
            nn.Linear(feature_dim, feature_dim),
            nn.SiLU(),
            nn.Linear(feature_dim, feature_dim)
        )

    def forward(self, x, edge_index, node_values):
        return self.propagate(edge_index, x=x, node_values=node_values)

    def message(self, x_i, x_j, node_values_i, node_values_j, index, ptr, size_i):
        tmp = torch.cat([x_i, x_j], dim=1)
        msg = self.mlp(tmp)
        
        # 1. Calculate the raw, unnormalized score (closer values = score closer to 0, further = more negative)
        raw_score = -torch.abs(node_values_i - node_values_j)
        
        # 2. Apply Softmax grouped by the destination node (index).
        # This guarantees that for any target node, the sum of incoming edge_weights == 1.0
        edge_weight = softmax(raw_score, index, ptr, size_i)
        return msg * edge_weight

class HierarchicalNetwork(nn.Module):
    def __init__(self, feature_dim):
        super(HierarchicalNetwork, self).__init__()
        # Shared modules across all levels
        self.upward_gnn = AgnosticUpwardModule(feature_dim)
        self.downward_gnn = AgnosticDownwardModule(feature_dim)

    def forward(self, x, edge_index, node_level_id, edge_level_id, combined_node_values):
        """
        x: [N, D] (assuming D is feature dimension, batched nodes)
        edge_index: [2, E]
        node_level_id: [N]
        edge_level_id: [E] (optional: can be used if edges have specific features to filter)
        """
        max_level = node_level_id.max().item()
        out_x = x.clone()

        # 1. UPWARD PASS (Bottom-to-Top)
        for l in range(max_level):
            # Find nodes at current level l and next level l+1
            source_mask = (node_level_id[edge_index[0]] == l)
            target_mask = (node_level_id[edge_index[1]] == l + 1)
            
            # Filter for edges going UP: from l to l+1
            up_edge_mask = source_mask & target_mask
            up_edge_index = edge_index[:, up_edge_mask]
            
            # Only update if there are edges connecting these levels
            if up_edge_index.size(1) > 0:
                # Calculate updates
                updated_features = self.upward_gnn(out_x, up_edge_index, combined_node_values)
                
                # Apply updates ONLY to the target nodes at level l+1
                # (Since PyTorch Geometric updates all nodes in the graph based on the 
                # edge index, we selectively overwrite only the target nodes)
                target_nodes = up_edge_index[1].unique()
                out_x[target_nodes] = updated_features[target_nodes]

        # 2. DOWNWARD PASS (Top-to-Bottom)
        for l in range(max_level, 0, -1):
            # Find nodes at current level l and lower level l-1
            source_mask = (node_level_id[edge_index[0]] == l)
            target_mask = (node_level_id[edge_index[1]] == l - 1)
            
            # Filter for edges going DOWN: from l to l-1
            down_edge_mask = source_mask & target_mask
            down_edge_index = edge_index[:, down_edge_mask]
            
            if down_edge_index.size(1) > 0:
                updated_features = self.downward_gnn(out_x, down_edge_index, combined_node_values)
                
                # Apply updates ONLY to the target nodes at level l-1
                target_nodes = down_edge_index[1].unique()
                out_x[target_nodes] = updated_features[target_nodes]

        return out_x
    
class Regressor(nn.Module):
    def __init__(self, feature_dim, output_dim=1):
        super(Regressor, self).__init__()
        self.mlp = nn.Sequential(
            nn.Linear(feature_dim, feature_dim),
            nn.SiLU(),
            nn.Linear(feature_dim, output_dim)
        )

    def forward(self, x):
        return self.mlp(x)
# =========================================================================
# Discrete Subgraph Generation (Non-diff) (only for visualization/analysis)
# =========================================================================

def generate_filtration_subgraphs(data, node_values, edge_values):
    """
    Constructs the actual NetworkX graphs for the filtration sequence.
    This part is for analysis/visualization and is not differentiable.
    For batched inputs, only graph 0 is visualized.
    """
    if node_values.dim() > 1:
        node_values = node_values.squeeze(-1)
    if edge_values.dim() > 1:
        edge_values = edge_values.squeeze(-1)

    edge_index = data.edge_index
    if hasattr(data, 'batch') and data.batch is not None:
        first_graph_mask = (data.batch == 0)
        if not first_graph_mask.any():
            return []

        first_nodes = torch.where(first_graph_mask)[0]
        node_offset = int(first_nodes[0].item())
        node_values = node_values[first_graph_mask]
        edge_mask = first_graph_mask[edge_index[0]] & first_graph_mask[edge_index[1]]
        edge_index = edge_index[:, edge_mask] - node_offset
        edge_values = edge_values[edge_mask]

    # 1. Collect all unique critical values (thresholds)
    # We combine node and edge values and sort them unique
    all_values = torch.cat([node_values.reshape(-1), edge_values.reshape(-1)])
    thresholds, _ = torch.sort(torch.unique(all_values))

    filtration_sequence = []

    print(f"Generating {len(thresholds)} filtration steps...")

    edges_np = edge_index.t().detach().cpu().numpy()
    for t in thresholds:
        # Identify nodes and edges that exist at time t
        # (Using <= allows the complex to grow)
        valid_edges_mask = (edge_values <= t).detach().cpu().numpy()

        # Get indices
        valid_node_indices = torch.where(node_values <= t)[0].tolist()

        # Create subgraph
        # Note: In a strict simplex filtration, we add nodes/edges one by one.
        # Here we add "batches" of elements that share the exact same scalar value.
        subg = nx.Graph()

        # Add only valid nodes
        for node_idx in valid_node_indices:
            subg.add_node(node_idx)

        # Add only valid edges
        # We iterate original edge_index to find valid ones
        # (Converting edge_index to list of pairs for iteration)
        for i, (u, v) in enumerate(edges_np):
            if valid_edges_mask[i]:
                # Only add edge if BOTH endpoints are already in subgraph
                # (The max condition technically guarantees this, but safety check)
                if subg.has_node(u) and subg.has_node(v):
                    subg.add_edge(u, v)

        filtration_sequence.append((t.item(), subg))

    return filtration_sequence
    

In [ ]:
def build_hierarchical_graph(node_features, edge_index, batch, node_values, edge_values, grouping_param=0.000):
    """
    Constructs a hierarchical graph structure based on edge values.

    Args:
        node_features (Tensor): [N, F]
        edge_index (Tensor): [2, E]
        batch (Tensor): [N]
        node_values (Tensor): [N, 1]
        edge_values (Tensor): [E, 1] OR [E]
        grouping_param (float): Grouping threshold parameter.

    Returns:
        combined_features (Tensor): [N_total, F]
        combined_edge_index (Tensor): [2, E_total]
        node_level_ids (Tensor): [N_total]
        edge_level_ids (Tensor): [E_total]
        combined_node_values (Tensor): [N_total, 1]
    """
    device = node_features.device
    num_nodes = node_features.shape[0]
    num_features = node_features.shape[1]

    # Flatten edge_values if needed
    if edge_values.dim() > 1:
        edge_values = edge_values.squeeze()

    # --- 1. Edge Sorting ---
    sorted_idx = torch.argsort(edge_values)
    sorted_values = edge_values[sorted_idx]
    sorted_u = edge_index[0, sorted_idx]
    sorted_v = edge_index[1, sorted_idx]

    # --- 2. Initialization ---
    all_features = [node_features]
    all_edges_list = []
    all_level_ids = [torch.zeros(num_nodes, dtype=torch.long, device=device)]
    all_edge_level_ids = []
    all_node_values = [node_values]

    # Maintains mapping from Original Node ID -> Current Cluster ID
    global_cluster_map = torch.arange(num_nodes, device=device)
    current_num_clusters = num_nodes
    current_node_values = node_values

    current_level = 0
    num_graphs = batch.max().item() + 1

    while current_num_clusters > 1:
        # --- 3. Identify Candidates ---
        c_u = global_cluster_map[sorted_u]
        c_v = global_cluster_map[sorted_v]

        diff_mask = c_u != c_v
        if not diff_mask.any():
            break

        valid_indices = torch.nonzero(diff_mask).squeeze()
        if valid_indices.dim() == 0:
            valid_indices = valid_indices.unsqueeze(0)

        first_valid_idx = valid_indices[0]
        min_cluster_diff_val = sorted_values[first_valid_idx]

        threshold = min_cluster_diff_val + grouping_param
        limit_idx = torch.searchsorted(sorted_values, threshold, right=True)

        cand_indices_mask = valid_indices < limit_idx
        merging_indices = valid_indices[cand_indices_mask]

        if merging_indices.numel() == 0:
            break

        merge_u = c_u[merging_indices]
        merge_v = c_v[merging_indices]

        # --- 4. Compute Connected Components (Next Layer) ---
        # Build an undirected sparse adjacency over current clusters and run exact CC.
        rows = torch.cat([merge_u, merge_v]).detach().cpu().numpy()
        cols = torch.cat([merge_v, merge_u]).detach().cpu().numpy()
        data = np.ones(rows.shape[0], dtype=np.int8)

        adj = sp.coo_matrix(
            (data, (rows, cols)),
            shape=(current_num_clusters, current_num_clusters)
        )

        num_new_clusters, labels_np = connected_components(
            adj,
            directed=False,
            return_labels=True
        )

        inverse_indices = torch.from_numpy(labels_np).to(device=device, dtype=torch.long)

        # inverse_indices maps current cluster IDs to their new parent cluster ID.
        new_layer_node_values = scatter(
            current_node_values,
            inverse_indices,
            dim=0,
            dim_size=num_new_clusters,
            reduce='mean'
        )
        all_node_values.append(new_layer_node_values)
        current_node_values = new_layer_node_values

        # --- 5. Build Next Layer ---
        new_layer_features = torch.zeros((num_new_clusters, num_features), device=device)
        all_features.append(new_layer_features)

        current_level += 1
        all_level_ids.append(torch.full((num_new_clusters,), current_level, dtype=torch.long, device=device))

        src_nodes = torch.arange(current_num_clusters, device=device)
        dst_nodes = inverse_indices
        all_edges_list.append((src_nodes, dst_nodes))

        edge_level = current_level - 1
        all_edge_level_ids.append(
            torch.full((src_nodes.size(0) * 2,), edge_level, dtype=torch.long, device=device)
        )

        global_cluster_map = inverse_indices[global_cluster_map]
        current_num_clusters = num_new_clusters

        if current_num_clusters <= num_graphs:
            break

    # --- 6. Assemble Final Graph ---
    layer_sizes = [f.shape[0] for f in all_features]
    cumulative_sizes = torch.tensor([0] + layer_sizes[:-1], device=device).cumsum(0)

    combined_features = torch.cat(all_features, dim=0)
    node_level_ids = torch.cat(all_level_ids, dim=0)
    combined_node_values = torch.cat(all_node_values, dim=0)

    final_edge_sources = []
    final_edge_targets = []

    for i, (src_local, dst_local) in enumerate(all_edges_list):
        offset_src = cumulative_sizes[i]
        offset_dst = cumulative_sizes[i + 1]

        u = src_local + offset_src
        v = dst_local + offset_dst

        final_edge_sources.append(u)
        final_edge_targets.append(v)
        final_edge_sources.append(v)
        final_edge_targets.append(u)

    if len(final_edge_sources) > 0:
        combined_edge_index = torch.stack([
            torch.cat(final_edge_sources),
            torch.cat(final_edge_targets)
        ], dim=0)
        edge_level_ids = torch.cat(all_edge_level_ids, dim=0)
    else:
        combined_edge_index = torch.empty((2, 0), dtype=torch.long, device=device)
        edge_level_ids = torch.empty((0,), dtype=torch.long, device=device)

    return combined_features, combined_edge_index, node_level_ids, edge_level_ids, combined_node_values

In [35]:
@torch.no_grad()
def get_components_gpu(edge_index, num_nodes):
    """
    Finds connected components entirely on the GPU using iterative label propagation.
    Instead of tracing pointers (which GPUs hate), we flood-fill minimum IDs across edges.
    """
    # Initialize every node's cluster label to its own index (0 to num_nodes - 1)
    labels = torch.arange(num_nodes, device=edge_index.device)
    
    # We must make edges bidirectional so the "flood fill" flows both ways.
    # .flip(0) flips the 2xE tensor upside down, swapping sources and targets.
    if edge_index.size(1) > 0:
        bidirectional_edges = torch.cat([edge_index, edge_index.flip(0)], dim=1)
    else:
        return labels

    changed = True
    while changed:
        prev_labels = labels.clone()
        src_labels = labels[bidirectional_edges[0]]
        
        # --- LESSER KNOWN FUNCTION: scatter_reduce_ ---
        # scatter_reduce_ takes values (src_labels) and pushes them into specific 
        # indices of a target tensor (labels) based on an index map (bidirectional_edges[1]).
        # reduce="amin" means if multiple values try to go into the same index, it keeps the minimum.
        # include_self=True ensures the node's current label is included in the minimum check.
        # This effectively "pulls" the smallest cluster ID from all neighbors.
        labels.scatter_reduce_(0, bidirectional_edges[1], src_labels, reduce="amin", include_self=True)
        
        # If no labels changed this iteration, the clusters have fully converged.
        changed = not torch.equal(labels, prev_labels)
        
    return labels


def _process_single_hierarchical_graph(node_features, edge_index, node_values, edge_values, grouping_param):
    """
    Internal helper to process a single, isolated disjoint graph.
    """
    device = node_features.device
    num_nodes = node_features.shape[0]
    num_features = node_features.shape[1]
    
    if edge_values.dim() > 1:
        edge_values = edge_values.squeeze()

    # --- 1. Edge Sorting ---
    # Sort all edges by their scalar value. 
    # argsort returns the indices that would sort the array, which we use to reorder the edges.
    sorted_idx = torch.argsort(edge_values)
    sorted_values = edge_values[sorted_idx]
    sorted_u = edge_index[0, sorted_idx]
    sorted_v = edge_index[1, sorted_idx]

    # --- 2. Initialization ---
    all_features = [node_features]
    all_edges_list = []
    all_level_ids = [torch.zeros(num_nodes, dtype=torch.long, device=device)]
    all_edge_level_ids = []
    all_node_values = [node_values]
    
    # This tracks which supernode every original node currently belongs to.
    global_cluster_map = torch.arange(num_nodes, device=device)
    current_num_clusters = num_nodes
    current_node_values = node_values
    current_level = 0
    
    while current_num_clusters > 1:
        # --- 3. Identify Candidates ---
        # Look up the current supernodes for the edges
        c_u = global_cluster_map[sorted_u]
        c_v = global_cluster_map[sorted_v]
        
        # We only care about edges that connect *different* clusters.
        diff_mask = c_u != c_v
        if not diff_mask.any():
            break 
            
        # torch.nonzero gets the indices where the mask is True.
        valid_indices = torch.nonzero(diff_mask).squeeze()
        if valid_indices.dim() == 0:
            valid_indices = valid_indices.unsqueeze(0)
            
        # The lowest edge value connecting two different clusters becomes our baseline.
        first_valid_idx = valid_indices[0]
        min_cluster_diff_val = sorted_values[first_valid_idx]
        threshold = min_cluster_diff_val + grouping_param
        
        # --- LESSER KNOWN FUNCTION: torch.searchsorted ---
        # Finds the index where 'threshold' should be inserted into 'sorted_values' 
        # to maintain the sorted order. It's an ultra-fast binary search (O(log N)).
        # It instantly tells us how many edges fall under our threshold.
        limit_idx = torch.searchsorted(sorted_values, threshold, right=True)
        
        cand_indices_mask = valid_indices < limit_idx
        merging_indices = valid_indices[cand_indices_mask]
        
        if merging_indices.numel() == 0:
            break
            
        merge_u = c_u[merging_indices]
        merge_v = c_v[merging_indices]
        
        # --- 4. Compute Connected Components (Next Layer) ---
        cluster_edges = torch.stack([merge_u, merge_v], dim=0)
        parent_map = get_components_gpu(cluster_edges, current_num_clusters)
        
        # --- LESSER KNOWN FUNCTION: torch.unique(return_inverse=True) ---
        # unique_parents: The distinct cluster IDs.
        # inverse_indices: A tensor of the same shape as parent_map. It contains the index 
        # in 'unique_parents' where each element of 'parent_map' can be found.
        # This effectively compacts our random cluster IDs (e.g., [0, 5, 8]) into a neat 
        # continuous sequence (e.g., [0, 1, 2]) mapping old clusters to new ones!
        unique_parents, inverse_indices = torch.unique(parent_map, return_inverse=True)
        num_new_clusters = unique_parents.size(0)
        
        # --- Native PyTorch Mean Pooling ---
        is_1d = current_node_values.dim() == 1
        vals_2d = current_node_values.unsqueeze(-1) if is_1d else current_node_values
        
        sums = torch.zeros(num_new_clusters, vals_2d.size(1), device=device, dtype=vals_2d.dtype)
        counts = torch.zeros(num_new_clusters, device=device, dtype=vals_2d.dtype)
        
        # scatter_add_ safely adds values from vals_2d into the 'sums' tensor based on 'inverse_indices'.
        idx_expanded = inverse_indices.unsqueeze(1).expand_as(vals_2d)
        sums.scatter_add_(0, idx_expanded, vals_2d)
        
        # We also count how many nodes merged into each supernode to divide for the mean.
        counts.scatter_add_(0, inverse_indices, torch.ones_like(inverse_indices, dtype=counts.dtype))
        
        # clamp(min=1) prevents division by zero.
        new_layer_node_values = sums / counts.unsqueeze(1).clamp(min=1)
        if is_1d:
            new_layer_node_values = new_layer_node_values.squeeze(-1)

        all_node_values.append(new_layer_node_values)
        current_node_values = new_layer_node_values 
        
        # --- 5. Build Next Layer ---
        # Supernode features are initialized to zero.
        new_layer_features = torch.zeros((num_new_clusters, num_features), device=device)
        all_features.append(new_layer_features)
        
        current_level += 1
        all_level_ids.append(torch.full((num_new_clusters,), current_level, dtype=torch.long, device=device))
        
        # Link the lower layer (src_nodes) to the new supernodes (dst_nodes).
        src_nodes = torch.arange(current_num_clusters, device=device)
        dst_nodes = inverse_indices
        all_edges_list.append((src_nodes, dst_nodes))
        
        edge_level = current_level - 1
        all_edge_level_ids.append(
            torch.full((src_nodes.size(0) * 2,), edge_level, dtype=torch.long, device=device)
        )
        
        # Advanced indexing: Update the tracking map so all base nodes now point to the new supernodes.
        global_cluster_map = inverse_indices[global_cluster_map]
        current_num_clusters = num_new_clusters

    # --- 6. Assemble Single Graph ---
    # Calculate cumulative offsets to shift the localized node IDs back into a single continuous tensor.
    # cumsum(0) calculates the running total (e.g., [2, 3, 5] -> [2, 5, 10]).
    layer_sizes = [f.shape[0] for f in all_features]
    cumulative_sizes = torch.tensor([0] + layer_sizes[:-1], device=device).cumsum(0)
    
    combined_features = torch.cat(all_features, dim=0)
    node_level_ids = torch.cat(all_level_ids, dim=0)
    combined_node_values = torch.cat(all_node_values, dim=0)
    
    final_edge_sources = []
    final_edge_targets = []
    
    for i, (src_local, dst_local) in enumerate(all_edges_list):
        offset_src = cumulative_sizes[i]
        offset_dst = cumulative_sizes[i+1]
        
        u = src_local + offset_src
        v = dst_local + offset_dst
        
        final_edge_sources.append(u)
        final_edge_targets.append(v)
        final_edge_sources.append(v) # Make edge bidirectional
        final_edge_targets.append(u)
        
    if len(final_edge_sources) > 0:
        combined_edge_index = torch.stack([
            torch.cat(final_edge_sources),
            torch.cat(final_edge_targets)
        ], dim=0)
        edge_level_ids = torch.cat(all_edge_level_ids, dim=0)
    else:
        combined_edge_index = torch.empty((2, 0), dtype=torch.long, device=device)
        edge_level_ids = torch.empty((0,), dtype=torch.long, device=device)
        
    return combined_features, combined_edge_index, node_level_ids, edge_level_ids, combined_node_values


def build_hierarchical_graph(node_features, edge_index, batch, node_values, edge_values, grouping_param=0.000, return_hier_batch=False):
    """
    Constructs a hierarchical graph structure dynamically handling multi-graph batches using CUDA streams.
    """
    device = node_features.device
    num_graphs = batch.max().item() + 1
    
    # --- LESSER KNOWN FUNCTION: torch.bincount ---
    # Counts the frequency of each non-negative integer in a tensor.
    # If batch is [0, 0, 1, 1, 1], bincount returns [2, 3], meaning 2 nodes in graph 0, 3 in graph 1.
    node_counts = torch.bincount(batch, minlength=num_graphs)
    node_offsets = torch.cat([torch.tensor([0], device=device), node_counts.cumsum(0)[:-1]])
    
    out_features = [None] * num_graphs
    out_edges = [None] * num_graphs
    out_node_levels = [None] * num_graphs
    out_edge_levels = [None] * num_graphs
    out_node_values = [None] * num_graphs
    
    # --- LESSER KNOWN FUNCTION: torch.cuda.Stream ---
    # A stream is a linear sequence of execution that belongs to a specific device.
    # Operations inside a single stream execute sequentially. Operations in DIFFERENT 
    # streams can execute concurrently (in parallel) on the GPU hardware.
    streams = [torch.cuda.Stream(device=device) for _ in range(num_graphs)]
    
    for i in range(num_graphs):
        # Assign the workload for Graph 'i' to its dedicated CUDA stream.
        with torch.cuda.stream(streams[i]):
            node_mask = batch == i
            if not node_mask.any():
                continue
                
            g_nodes = node_features[node_mask]
            g_node_vals = node_values[node_mask]
            
            edge_mask = node_mask[edge_index[0]]
            
            # Shift edge indices down so Graph 'i' is indexed starting from 0,
            # allowing it to be processed as an isolated graph.
            g_edges = edge_index[:, edge_mask] - node_offsets[i]
            g_edge_vals = edge_values[edge_mask]
            
            f, e, nl, el, nv = _process_single_hierarchical_graph(
                g_nodes, g_edges, g_node_vals, g_edge_vals, grouping_param
            )
            
            out_features[i] = f
            out_edges[i] = e
            out_node_levels[i] = nl
            out_edge_levels[i] = el
            out_node_values[i] = nv

    # --- LESSER KNOWN FUNCTION: torch.cuda.synchronize ---
    # Because the streams are running asynchronously, the main Python script will hit this 
    # line before the GPU has actually finished the math. 
    # synchronize() acts as a roadblock, forcing the CPU to wait here until ALL active 
    # GPU streams have completely finished their work.
    torch.cuda.synchronize(device=device)
    
    valid_indices = [i for i, f in enumerate(out_features) if f is not None]
    
    # Re-batch sequentially. Since we know the final shapes of everything now,
    # this is just simple concatenation and pointer math.
    combined_features = []
    combined_edges = []
    combined_batch_ids = []
    cumulative_offset = 0
    
    for i in valid_indices:
        combined_features.append(out_features[i])
        combined_batch_ids.append(torch.full((out_features[i].shape[0],), i, dtype=torch.long, device=device))
        
        e = out_edges[i]
        if e.size(1) > 0:
            # Shift the edge connections up by the total number of nodes added to the master tensor so far.
            combined_edges.append(e + cumulative_offset)
            
        cumulative_offset += out_features[i].shape[0]

    final_features = torch.cat(combined_features, dim=0)
    final_node_levels = torch.cat([out_node_levels[i] for i in valid_indices], dim=0)
    final_node_values = torch.cat([out_node_values[i] for i in valid_indices], dim=0)
    final_hier_batch = torch.cat(combined_batch_ids, dim=0)
    
    if combined_edges:
        final_edge_index = torch.cat(combined_edges, dim=1)
        final_edge_levels = torch.cat([out_edge_levels[i] for i in valid_indices], dim=0)
    else:
        final_edge_index = torch.empty((2, 0), dtype=torch.long, device=device)
        final_edge_levels = torch.empty((0,), dtype=torch.long, device=device)

    if return_hier_batch:
        return final_features, final_edge_index, final_node_levels, final_edge_levels, final_node_values, final_hier_batch
    return final_features, final_edge_index, final_node_levels, final_edge_levels, final_node_values

In [36]:
class TPNN(torch.nn.Module):
    def __init__(self, hidden_channels, output_channels=1, grouping_param=0.000,
                 pretrained_root=None, pretrained_dataset=None, pretrained_target=None,
                 freeze_encoder=False):
        super().__init__()
        self.grouping_param = grouping_param
        # 1. Base Encoder: DimeNet++
        if pretrained_root is not None and pretrained_dataset is not None and pretrained_target is not None:
            # Load pretrained encoder from QM9
            self.encoder, _ = DimeNetPlusPlusNode.from_qm9_pretrained(
                pretrained_root, pretrained_dataset, pretrained_target
            )
            # Pretrained uses hidden_channels=128
            hidden_channels = 128
            print(f"Loaded pretrained DimeNet++ encoder (target={pretrained_target})")
            
            if freeze_encoder:
                for param in self.encoder.parameters():
                    param.requires_grad = False
                print("Encoder weights frozen")
        else:
            # Train from scratch
            self.encoder = DimeNetPlusPlusNode(
                hidden_channels=hidden_channels,
                out_channels=1,  # out_channels unused in our forward (node embeddings)
                num_blocks=3,
                int_emb_size=64,
                basis_emb_size=8,
                out_emb_channels=256,
                num_spherical=7,
                num_radial=6,
                cutoff=5.0,
                max_num_neighbors=32,
                envelope_exponent=5,
                num_before_skip=1,
                num_after_skip=2,
                num_output_layers=3,
                output_initializer='glorot_orthogonal',
            )
            
        self.kernel = ScalarKernel(in_channels=hidden_channels, hidden_channels=hidden_channels, out_channels=output_channels)
        
        self.HierMP = HierarchicalNetwork(feature_dim=hidden_channels)
        
        self.regressor = Regressor(feature_dim=hidden_channels, output_dim=output_channels)
    
    def forward(self, batch):
        
 
        x = self.encoder(batch.z, batch.pos, batch.batch)  # Get node embeddings from DimeNet++
        
        node_values, edge_values = self.kernel(x, batch.edge_index)  # Apply kernel to node embeddings
        
        combined_features, combined_edge_index, node_level_ids, edge_level_ids, combined_node_values, hier_batch = build_hierarchical_graph(
            x, batch.edge_index, batch.batch, node_values, edge_values,
            grouping_param=self.grouping_param, return_hier_batch=True
        )

        x_post = self.HierMP(combined_features, combined_edge_index, node_level_ids, edge_level_ids, combined_node_values)
        # last_node_index = torch.argmax(node_level_ids).item()
        # last_node_features = x_post[last_node_index,:]
        num_graphs = int(batch.batch.max().item() + 1)
        top_node_indices = []
        for b in range(num_graphs):
            graph_mask = (hier_batch == b)
            graph_indices = torch.where(graph_mask)[0]
            graph_node_levels = node_level_ids[graph_indices]
            top_local_idx = torch.argmax(graph_node_levels)
            top_node_indices.append(graph_indices[top_local_idx])

        top_node_indices = torch.stack(top_node_indices)
        top_node_features = x_post[top_node_indices]
        output = self.regressor(top_node_features)
        
        return x, x_post, node_values, edge_values, output  # Return node embeddings without global pooling
    

In [33]:
# --- Option A: Load pretrained DimeNet++ from QM9 ---
# target: 0=mu, 1=alpha, 2=HOMO, 3=LUMO, 5=R2, 6=ZPVE, 7=U0, 8=U, 9=H, 10=G, 11=Cv
# (target=4 is not available for pretrained models)
PRETRAINED_TARGET = 7
PRETRAINED_ROOT = os.path.expanduser("~/TPNN/dimenetpp-qm9")
ATOM_MAP_VIZ = {1: 'H', 6: 'C', 7: 'N', 8: 'O', 9: 'F', 16: 'S', 17: 'Cl'}

def test_pipeline(batch, tpnn_model):
    
    print("Testing pipeline with batch of size:", batch.num_graphs)
    print("Batch node feature shape:", batch.z.shape)
    print("Batch position shape:", batch.pos.shape)
    print("Batch edge index shape:", batch.edge_index.shape)
    print("Batch batch vector shape:", batch.batch.shape)
    print("Batch num nodes:", batch.num_nodes)
    
    tpnn_model.eval()
    with torch.no_grad():
        x, x_post,node_values, edge_values, output= tpnn_model(batch)
    
    print("Node embeddings shape:", x.shape)
    print("Node scalar values shape:", node_values.shape)
    print("Edge scalar values shape:", edge_values.shape)
    print("Post-Hierarchical node features shape:", x_post.shape)
    print("Output shape:", output.shape)

    
    return x, x_post, node_values, edge_values, output

batch = next(iter(train_loader))
batch = batch.to(device)
x, x_post, node_values, edge_values, output= test_pipeline(batch, TPNN(hidden_channels=128, grouping_param=0.01, pretrained_root=PRETRAINED_ROOT, pretrained_dataset=dataset, pretrained_target=PRETRAINED_TARGET, freeze_encoder=True).to(device))

combined_node_features, combined_edge_index, node_ids, edge_ids, combined_node_values = build_hierarchical_graph(x, batch.edge_index, batch.batch, node_values, edge_values)
print("Combined node features shape:", combined_node_features.shape)
print("Combined edge index shape:", combined_edge_index.shape)  
print("Node level IDs shape:", node_ids.shape)
print("Edge level IDs shape:", edge_ids.shape)
print("combined node values shape:", combined_node_values.shape)
print("Unique node levels:", torch.unique(node_ids))
print("Unique edge levels:", torch.unique(edge_ids))

# Build atom labels for this batch
z_np = batch.z.cpu().numpy()
_atom_counts = {}
unique_labels_viz = {}
for i in range(len(z_np)):
    sym = ATOM_MAP_VIZ.get(z_np[i], str(z_np[i]))
    _atom_counts[sym] = _atom_counts.get(sym, 0) + 1
    unique_labels_viz[i] = f"{sym}{_atom_counts[sym]}"

subgraphs = generate_filtration_subgraphs(batch, node_values, edge_values)

# E. Visualize All steps
steps_to_plot = [i for i in range(len(subgraphs))]
fig, axes = plt.subplots(1, len(steps_to_plot), figsize=(len(steps_to_plot)*2, 4))
pos = nx.spring_layout(to_networkx(batch), seed=42) # Fixed layout for consistency

for i, step_idx in enumerate(steps_to_plot):
    thresh, G = subgraphs[step_idx]
    ax = axes[i]
    
    # Draw the full "background" graph faintly
    nx.draw(to_networkx(batch), pos, ax=ax, alpha=0.1, node_color='grey')
    
    # Draw the active filtration subgraph with atom labels
    if len(G.nodes) > 0:
        sub_labels = {n: unique_labels_viz.get(n, str(n)) for n in G.nodes()}
        nx.draw(G, pos, ax=ax, with_labels=True, labels=sub_labels,
                node_color='red', edge_color='black')
    
    ax.set_title(f"Step {step_idx}\nThreshold: {thresh:.4f}")

plt.show()

Loaded pretrained DimeNet++ encoder (target=7)
Encoder weights frozen
Testing pipeline with batch of size: 2
Batch node feature shape: torch.Size([20])
Batch position shape: torch.Size([20, 3])
Batch edge index shape: torch.Size([2, 40])
Batch batch vector shape: torch.Size([20])
Batch num nodes: 20
Node embeddings shape: torch.Size([20, 128])
Node scalar values shape: torch.Size([20, 1])
Edge scalar values shape: torch.Size([40, 1])
Post-Hierarchical node features shape: torch.Size([34, 128])
Output shape: torch.Size([2, 1])
Combined node features shape: torch.Size([88, 128])
Combined edge index shape: torch.Size([2, 172])
Node level IDs shape: torch.Size([88])
Edge level IDs shape: torch.Size([172])
combined node values shape: torch.Size([88, 1])
Unique node levels: tensor([0, 1, 2, 3, 4, 5, 6, 7], device='cuda:0')
Unique edge levels: tensor([0, 1, 2, 3, 4, 5, 6], device='cuda:0')
Generating 20 filtration steps...


In [21]:
def visualize_hierarchy(batch, combined_edge_index, node_level_ids):
    """
    Visualizes the molecular structure alongside the constructed hierarchical graph.
    For batched inputs, only the first graph (graph index 0) is visualized.

    Args:
        batch: PyG batch object containing the molecule (pos, z, edge_index).
        combined_edge_index (Tensor): Edges of the full hierarchy.
        node_level_ids (Tensor): Level assignment for each node in hierarchy.
    """
    # Atom Dictionary
    ATOM_MAP = {1: 'H', 6: 'C', 7: 'N', 8: 'O', 9: 'F', 16: 'S', 17: 'Cl'}

    # Restrict molecular view to graph 0.
    if hasattr(batch, 'batch') and batch.batch is not None:
        first_graph_mask = (batch.batch == 0)
        if not first_graph_mask.any():
            raise ValueError('Cannot visualize: no graph with index 0 found in batch.')

        first_nodes = torch.where(first_graph_mask)[0]
        node_offset = int(first_nodes[0].item())

        z = batch.z[first_graph_mask].cpu().numpy()
        pos = batch.pos[first_graph_mask].cpu().numpy()

        mol_edge_mask = first_graph_mask[batch.edge_index[0]] & first_graph_mask[batch.edge_index[1]]
        mol_edges = (batch.edge_index[:, mol_edge_mask] - node_offset).t().cpu().numpy()
    else:
        z = batch.z.cpu().numpy()
        pos = batch.pos.cpu().numpy()
        mol_edges = batch.edge_index.t().cpu().numpy()

    # Restrict hierarchy view to graph 0 in the sequentially concatenated hierarchy.
    total_hier_nodes = int(node_level_ids.numel())
    if total_hier_nodes == 0:
        raise ValueError('Cannot visualize empty hierarchy.')

    seed_nodes = list(range(min(len(z), total_hier_nodes)))
    if not seed_nodes:
        raise ValueError('Cannot visualize hierarchy: first graph has no base nodes.')

    if combined_edge_index.numel() > 0:
        src_list = combined_edge_index[0].detach().cpu().tolist()
        dst_list = combined_edge_index[1].detach().cpu().tolist()

        adjacency = [[] for _ in range(total_hier_nodes)]
        for u, v in zip(src_list, dst_list):
            adjacency[u].append(v)
            adjacency[v].append(u)

        stack = seed_nodes.copy()
        selected_nodes = set(seed_nodes)
        while stack:
            u = stack.pop()
            for v in adjacency[u]:
                if v not in selected_nodes:
                    selected_nodes.add(v)
                    stack.append(v)
        selected_nodes = sorted(selected_nodes)
    else:
        selected_nodes = seed_nodes

    node_remap = {old_idx: new_idx for new_idx, old_idx in enumerate(selected_nodes)}
    selected_set = set(selected_nodes)
    levels = node_level_ids[selected_nodes].cpu().numpy()

    hier_edges = []
    if combined_edge_index.numel() > 0:
        src_np = combined_edge_index[0].detach().cpu().numpy()
        dst_np = combined_edge_index[1].detach().cpu().numpy()
        for u, v in zip(src_np, dst_np):
            u = int(u)
            v = int(v)
            if u in selected_set and v in selected_set:
                hier_edges.append((node_remap[u], node_remap[v]))

    # Generate Unique Labels (e.g., C1, C2, H1, H2...)
    atom_counts = {}
    unique_labels = {}
    for i in range(len(z)):
        atom_type = z[i]
        symbol = ATOM_MAP.get(atom_type, str(atom_type))
        if symbol not in atom_counts:
            atom_counts[symbol] = 0
        atom_counts[symbol] += 1
        unique_labels[i] = f"{symbol}{atom_counts[symbol]}"

    # Setup Figure
    fig, axes = plt.subplots(1, 2, figsize=(20, 10))

    # -----------------------------
    # Left Plot: Molecular Structure
    # -----------------------------
    ax_mol = axes[0]
    G_mol = nx.Graph()
    G_mol.add_nodes_from(range(len(z)))
    G_mol.add_edges_from(mol_edges.tolist())

    # Layout: use spring_layout seeded from 2D projection to avoid overlapping atoms
    pos_init = {i: pos[i, :2] for i in range(len(z))}
    pos_mol = nx.spring_layout(G_mol, pos=pos_init, seed=42, k=2.0/np.sqrt(len(z)), iterations=100)

    colors = ['#FF9999' if z[i]==8 else '#9999FF' if z[i]==7 else '#90EE90' if z[i]==6 else '#E0E0E0' for i in range(len(z))]

    nx.draw(G_mol, pos_mol, ax=ax_mol, with_labels=True, labels=unique_labels,
            node_color=colors, node_size=800, edge_color='gray', width=2, font_size=10, font_weight='bold')
    ax_mol.set_title("Input Molecular Structure (2D Projection)", fontsize=14)
    ax_mol.axis('off')

    # -----------------------------
    # Right Plot: Hierarchy
    # -----------------------------
    ax_hier = axes[1]
    G_hier = nx.Graph()
    G_hier.add_nodes_from(range(len(levels)))
    G_hier.add_edges_from(hier_edges)

    # --- Compute Layout ---
    # Strategy: Place nodes at each level equally spaced.

    pos_hier = {}

    # Group nodes by level
    nodes_by_level = {}
    for i, l in enumerate(levels):
        if l not in nodes_by_level: nodes_by_level[l] = []
        nodes_by_level[l].append(i)

    sorted_levels = sorted(nodes_by_level.keys())

    # Equal spacing for ALL levels
    for l in sorted_levels:
        level_nodes = sorted(nodes_by_level[l])
        n_nodes = len(level_nodes)
        for idx, node_id in enumerate(level_nodes):
            norm_x = (idx / max(1, n_nodes - 1)) * 10 if n_nodes > 1 else 5.0
            pos_hier[node_id] = (norm_x, float(l))

    # --- Draw Hierarchy ---

    # Node Colors by Level (cmap)
    node_colors = [levels[n] for n in G_hier.nodes()]

    # Node Labels: Only for Level 0
    hier_labels = {}
    for n in G_hier.nodes():
        if levels[n] == 0:
            if n < len(z):
                hier_labels[n] = unique_labels.get(n, '?')
            else:
                hier_labels[n] = '?'
        else:
            hier_labels[n] = ''

    # Draw edges first
    nx.draw_networkx_edges(G_hier, pos_hier, ax=ax_hier, alpha=0.3, edge_color='gray')

    # Draw nodes
    nx.draw_networkx_nodes(G_hier, pos_hier, ax=ax_hier,
                                node_color=node_colors, cmap=plt.cm.coolwarm,
                                node_size=600, edgecolors='black')

    # Draw labels (font size slightly smaller for indexed labels)
    nx.draw_networkx_labels(G_hier, pos_hier, ax=ax_hier, labels=hier_labels, font_size=8, font_weight='bold')

    # Add level lines (horizontal grid)
    max_level = levels.max()
    for l in range(max_level + 1):
        ax_hier.axhline(y=l, color='gray', linestyle='--', alpha=0.1, zorder=-1)
        ax_hier.text(-1, l, f"L{l}", verticalalignment='center', fontweight='bold', alpha=0.5)

    ax_hier.set_title(f"Hierarchical Grouping ({max_level+1} Levels)", fontsize=14)
    ax_hier.axis('off')

    plt.tight_layout()
    plt.show()

# Run Visualization on the previously computed result
visualize_hierarchy(batch, combined_edge_index, node_ids)

In [37]:
import os
import shutil
import glob
import contextlib
import io
import imageio.v2 as imageio
import matplotlib
matplotlib.use('Agg')  # Non-interactive backend for saving
import matplotlib.pyplot as plt

# ===================== Configuration =====================
TARGET_IDX = 11
PRETRAINED_TARGET = TARGET_IDX
PRETRAINED_ROOT = os.path.expanduser("~/TPNN/dimenetpp-qm9")
EPOCHS = 50
LR = 1e-4
VIZ_DIR = "training_viz_v2"
HIER_VIZ_EVERY = 2  # Only controls hierarchy/filtration image frequency
CLEAN_BEFORE_RUN = True
CLEAN_AFTER_RUN = True
KEEP_FINAL_ARTIFACTS = True
ATOM_MAP = {1: 'H', 6: 'C', 7: 'N', 8: 'O', 9: 'F', 16: 'S', 17: 'Cl'}


def clear_artifacts_before_run(save_dir):
    """Remove leftover artifacts from previous runs and recreate output folder."""
    if os.path.exists(save_dir):
        shutil.rmtree(save_dir)
    os.makedirs(save_dir, exist_ok=True)


def cleanup_artifacts_after_run(save_dir, keep_files=None):
    """Clear artifacts after run, optionally keeping selected final outputs."""
    if not os.path.isdir(save_dir):
        return

    keep = set(keep_files or [])
    for path in glob.glob(os.path.join(save_dir, "*")):
        name = os.path.basename(path)
        if name in keep:
            continue
        if os.path.isdir(path):
            shutil.rmtree(path)
        else:
            os.remove(path)


def compute_non_encoder_grad_norms(model):
    """Compute per-top-module gradient norm, excluding encoder parameters."""
    sq_norms = {}
    for name, param in model.named_parameters():
        if name.startswith("encoder.") or param.grad is None:
            continue

        grad = param.grad.detach()
        if not torch.is_floating_point(grad):
            grad = grad.float()

        module_name = name.split(".")[0]
        gnorm = torch.linalg.norm(grad).item()
        sq_norms[module_name] = sq_norms.get(module_name, 0.0) + gnorm * gnorm

    return {k: v ** 0.5 for k, v in sq_norms.items()}


def plot_gradient_history(grad_history, save_dir):
    """Plot epoch-wise gradient norms for all non-encoder modules."""
    if not grad_history:
        print("No non-encoder gradients recorded. Skipping gradient plot.")
        return

    fig, ax = plt.subplots(1, 1, figsize=(12, 5))
    for module_name in sorted(grad_history.keys()):
        ax.plot(grad_history[module_name], label=module_name)

    ax.set_xlabel("Epoch")
    ax.set_ylabel("Gradient L2 Norm")
    ax.set_title("Gradient Norms by Module (Encoder Excluded)")
    ax.grid(True, alpha=0.3)
    ax.legend(loc="upper right")

    plt.tight_layout()
    plt.savefig(os.path.join(save_dir, "gradient_norms.png"), dpi=120)
    plt.close(fig)


if CLEAN_BEFORE_RUN:
    clear_artifacts_before_run(VIZ_DIR)
else:
    os.makedirs(VIZ_DIR, exist_ok=True)


# ===================== Target Normalization (z-score from training set) =====================
class TargetNormalizer:
    """Z-score normalizer fitted on training data only."""

    def __init__(self, train_dataset, target_idx):
        targets = torch.tensor([d.y[0, target_idx].item() for d in train_dataset])
        self.mean = targets.mean().item()
        self.std = targets.std().item()
        # Guard against zero std (constant target)
        if self.std < 1e-12:
            self.std = 1.0
        print(f"TargetNormalizer | target={target_idx} | mean={self.mean:.6f} | std={self.std:.6f}")

    def normalize(self, y):
        """Transform raw target to z-score."""
        return (y - self.mean) / self.std

    def inverse(self, y_norm):
        """Transform z-score back to original scale."""
        return y_norm * self.std + self.mean


# normalizer = TargetNormalizer(train_dataset, TARGET_IDX)


# ===================== Visualization Helper =====================
def save_epoch_viz(batch, node_values, edge_values, combined_edge_index,
                   node_level_ids, epoch, train_loss, val_loss, save_dir):
    """Save hierarchy + filtration visualization for a single graph to disk."""
    z = batch.z.cpu().numpy()
    pos_3d = batch.pos.cpu().numpy()
    levels = node_level_ids.cpu().numpy()
    edges = combined_edge_index.cpu().numpy().T

    # Atom labels
    atom_counts = {}
    unique_labels = {}
    for i in range(len(z)):
        symbol = ATOM_MAP.get(z[i], str(z[i]))
        atom_counts[symbol] = atom_counts.get(symbol, 0) + 1
        unique_labels[i] = f"{symbol}{atom_counts[symbol]}"

    # Filtration subgraphs (suppress prints)
    with contextlib.redirect_stdout(io.StringIO()):
        subgraphs = generate_filtration_subgraphs(batch, node_values, edge_values)
    n_steps = len(subgraphs)

    # Pick evenly-spaced filtration steps for bottom row
    n_filt_cols = min(n_steps, 8)
    step_indices = np.linspace(0, n_steps - 1, n_filt_cols, dtype=int) if n_steps > 0 else []
    n_filt_cols = max(n_filt_cols, 1)  # at least 1 col so subplot grid is valid

    fig = plt.figure(figsize=(max(20, n_filt_cols * 3), 14))
    fig.suptitle(f"Epoch {epoch} | Train Loss: {train_loss:.6f} | Val Loss: {val_loss:.6f}",
                 fontsize=16, fontweight='bold')

    # ---- Top Left: Molecular Structure ----
    ax_mol = fig.add_subplot(2, 2, 1)
    G_mol = to_networkx(batch, to_undirected=True)
    # Use spring_layout seeded from 2D projection to avoid overlapping atoms
    pos_init = {i: pos_3d[i, :2] for i in range(len(z))}
    pos_mol = nx.spring_layout(G_mol, pos=pos_init, seed=42, k=2.0 / np.sqrt(len(z)), iterations=100)
    colors_mol = [
        '#FF9999' if z[i] == 8 else
        '#9999FF' if z[i] == 7 else
        '#90EE90' if z[i] == 6 else '#E0E0E0'
        for i in range(len(z))
    ]
    nx.draw(G_mol, pos_mol, ax=ax_mol, with_labels=True, labels=unique_labels,
            node_color=colors_mol, node_size=800, edge_color='gray', width=2,
            font_size=10, font_weight='bold')
    ax_mol.set_title("Molecular Structure", fontsize=12)
    ax_mol.axis('off')

    # ---- Top Right: Hierarchy ----
    ax_hier = fig.add_subplot(2, 2, 2)
    G_hier = nx.Graph()
    G_hier.add_edges_from(edges)
    G_hier.add_nodes_from(range(len(levels)))

    # Layout: equally space all nodes at each level
    pos_hier = {}
    nodes_by_level = {}
    for i, l in enumerate(levels):
        nodes_by_level.setdefault(l, []).append(i)

    for l in sorted(nodes_by_level.keys()):
        level_nodes = sorted(nodes_by_level[l])
        n_nodes = len(level_nodes)
        for idx, nid in enumerate(level_nodes):
            norm_x = (idx / max(1, n_nodes - 1)) * 10 if n_nodes > 1 else 5.0
            pos_hier[nid] = (norm_x, float(l))

    node_colors_hier = [levels[n] for n in G_hier.nodes()]
    hier_labels = {n: (unique_labels.get(n, '') if levels[n] == 0 else '') for n in G_hier.nodes()}

    nx.draw_networkx_edges(G_hier, pos_hier, ax=ax_hier, alpha=0.3, edge_color='gray')
    nx.draw_networkx_nodes(G_hier, pos_hier, ax=ax_hier, node_color=node_colors_hier,
                           cmap=plt.cm.coolwarm, node_size=600, edgecolors='black')
    nx.draw_networkx_labels(G_hier, pos_hier, ax=ax_hier, labels=hier_labels,
                            font_size=8, font_weight='bold')

    max_level = int(levels.max())
    for l in range(max_level + 1):
        ax_hier.axhline(y=l, color='gray', linestyle='--', alpha=0.1, zorder=-1)
        ax_hier.text(-1, l, f"L{l}", verticalalignment='center', fontweight='bold', alpha=0.5)

    ax_hier.set_title(f"Hierarchy ({max_level + 1} Levels)", fontsize=12)
    ax_hier.axis('off')

    # ---- Bottom Row: Filtration Steps ----
    if len(step_indices) > 0:
        # Use the same spring layout as the molecular structure for consistency
        pos_spring = nx.spring_layout(G_mol, pos=pos_init, seed=42, k=2.0 / np.sqrt(len(z)), iterations=100)
        for col, si in enumerate(step_indices):
            ax = fig.add_subplot(2, n_filt_cols, n_filt_cols + col + 1)
            thresh, G_sub = subgraphs[si]
            nx.draw(G_mol, pos_spring, ax=ax, alpha=0.1, node_color='grey', node_size=100)
            if len(G_sub.nodes) > 0:
                # Use unique_labels so filtration nodes match molecular graph labels
                sub_labels = {n: unique_labels.get(n, str(n)) for n in G_sub.nodes()}
                nx.draw(G_sub, pos_spring, ax=ax, with_labels=True, labels=sub_labels,
                        node_color='red', edge_color='black', node_size=200, font_size=6)
            ax.set_title(f"t={thresh:.3f}", fontsize=9)
            ax.axis('off')

    plt.tight_layout()
    plt.savefig(os.path.join(save_dir, f"epoch_{epoch:03d}.png"), dpi=100, bbox_inches='tight')
    plt.close(fig)


# ===================== Model Setup =====================
model = TPNN(
    hidden_channels=128,
    output_channels=1,
    grouping_param=0.000,
    pretrained_root=PRETRAINED_ROOT,
    pretrained_dataset=dataset,
    pretrained_target=PRETRAINED_TARGET,
    freeze_encoder=False
).to(device)

optimizer = torch.optim.Adam(filter(lambda p: p.requires_grad, model.parameters()), lr=LR)
criterion = nn.MSELoss()

# Grab only the first graph from the first training batch for per-epoch visualization.
first_train_batch = next(iter(train_loader))
viz_batch = first_train_batch.to_data_list()[0]
viz_batch.batch = torch.zeros(viz_batch.num_nodes, dtype=torch.long)
viz_batch = viz_batch.to(device)

# ===================== Training Loop =====================
print(f"Training TPNN | Target: {TARGET_IDX} | Epochs: {EPOCHS} | LR: {LR}")
print(f"Train: {len(train_dataset)} | Val: {len(val_dataset)} | Test: {len(test_dataset)}")
print(f"Hierarchy/filtration visualization interval: every {HIER_VIZ_EVERY} epoch(s)")
print("Gradient logging interval: every epoch (fixed)")
print("-" * 60)

train_losses = []
val_losses = []
val_maes = []
grad_history = {}
tracked_modules = set()

interrupted = False
run_error = None

try:
    for epoch in range(EPOCHS):
        # --- Train ---
        model.train()
        epoch_loss = 0.0
        num_batches = 0
        epoch_grad_sums = {}
        epoch_grad_counts = {}

        for batch_data in train_loader:
            batch_data = batch_data.to(device)
            optimizer.zero_grad()

            x, x_post, nv, ev, pred = model(batch_data)
            target_raw = batch_data.y[:, TARGET_IDX]
            loss = criterion(pred.view_as(target_raw), target_raw)
            loss.backward()

            # Gradient tracking is fixed: always collected every epoch.
            batch_grad_norms = compute_non_encoder_grad_norms(model)
            for module_name, grad_norm in batch_grad_norms.items():
                epoch_grad_sums[module_name] = epoch_grad_sums.get(module_name, 0.0) + grad_norm
                epoch_grad_counts[module_name] = epoch_grad_counts.get(module_name, 0) + 1
                tracked_modules.add(module_name)

            optimizer.step()

            epoch_loss += loss.item()
            num_batches += 1

        avg_train_loss = epoch_loss / max(num_batches, 1)
        train_losses.append(avg_train_loss)

        # Save one gradient value per module per epoch (mean over train batches)
        for module_name in sorted(tracked_modules):
            grad_history.setdefault(module_name, [])
            if module_name in epoch_grad_sums:
                mean_grad = epoch_grad_sums[module_name] / max(epoch_grad_counts[module_name], 1)
            else:
                mean_grad = 0.0
            grad_history[module_name].append(mean_grad)

        # --- Validation ---
        model.eval()
        val_loss_total = 0.0
        val_mae_total = 0.0
        val_n_samples = 0
        val_batches = 0
        with torch.no_grad():
            for vb in val_loader:
                vb = vb.to(device)
                _, _, _, _, vpred = model(vb)
                vtarget_raw = vb.y[:, TARGET_IDX]

                val_loss_total += criterion(vpred.view_as(vtarget_raw), vtarget_raw).item()
                val_batches += 1

                val_mae_total += (vpred - vtarget_raw).abs().sum().item()
                val_n_samples += vtarget_raw.numel()

        avg_val_loss = val_loss_total / max(val_batches, 1)
        avg_val_mae = val_mae_total / max(val_n_samples, 1)
        val_losses.append(avg_val_loss)
        val_maes.append(avg_val_mae)

        # --- Visualization (interval-based, always include final epoch) ---
        if (epoch % HIER_VIZ_EVERY == 0) or (epoch == EPOCHS - 1):
            with torch.no_grad():
                x_viz, x_post_viz, nv_viz, ev_viz, out_viz = model(viz_batch)
                cf, cei, nli, eli, cnv = build_hierarchical_graph(
                    x_viz, viz_batch.edge_index, viz_batch.batch, nv_viz, ev_viz
                )
                save_epoch_viz(
                    viz_batch,
                    nv_viz,
                    ev_viz,
                    cei,
                    nli,
                    epoch,
                    avg_train_loss,
                    avg_val_loss,
                    VIZ_DIR
                )

        print(
            f"Epoch {epoch:03d} | Train Loss: {avg_train_loss:.6f} | "
            f"Val Loss: {avg_val_loss:.6f} | Val MAE: {avg_val_mae:.6f}"
        )

    print("-" * 60)
    print("Training complete!")

except KeyboardInterrupt:
    interrupted = True
    print("Training interrupted by user. Finalizing available logs/visuals and stopping...")

# except Exception as e:
#     run_error = e
#     print(f"Training stopped due to error: {e}")
#     print("Finalizing available logs/visuals from completed epochs...")

# ===================== Finalize Artifacts (works for full or interrupted runs) =====================
image_files = sorted(glob.glob(os.path.join(VIZ_DIR, "epoch_*.png")))
if image_files:
    images = [imageio.imread(f) for f in image_files]
    gif_path = os.path.join(VIZ_DIR, "training_evolution.gif")
    if os.path.exists(gif_path):
        os.remove(gif_path)
    imageio.mimsave(gif_path, images, duration=0.5)
    print(f"GIF saved: {gif_path} ({len(images)} frames)")
else:
    print("No hierarchy/filtration images found to create GIF.")

if len(train_losses) > 0 and len(val_losses) > 0 and len(val_maes) > 0:
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 5))

    ax1.plot(train_losses, label='Train Loss')
    ax1.plot(val_losses, label='Val Loss')
    ax1.set_xlabel('Epoch')
    ax1.set_ylabel('MSE Loss')
    ax1.set_title('Training Progress')
    ax1.legend()
    ax1.grid(True, alpha=0.3)

    ax2.plot(val_maes, label='Val MAE', color='tab:orange')
    ax2.set_xlabel('Epoch')
    ax2.set_ylabel('MAE')
    ax2.set_title('Validation MAE')
    ax2.legend()
    ax2.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.savefig(os.path.join(VIZ_DIR, "loss_curves.png"), dpi=100)
    plt.close(fig)
    print("Loss curves saved.")
else:
    print("Insufficient loss history to plot loss_curves.png")

plot_gradient_history(grad_history, VIZ_DIR)

# Always stop here after finalization if interrupted or error happened.
if interrupted or (run_error is not None):
    print("Run stopped early after artifact finalization.")

# ===================== Cleanup =====================
if CLEAN_AFTER_RUN:
    keep = []
    if KEEP_FINAL_ARTIFACTS:
        keep = ["training_evolution.gif", "loss_curves.png", "gradient_norms.png"]
    cleanup_artifacts_after_run(VIZ_DIR, keep_files=keep)
    print(f"Post-run cleanup completed for {VIZ_DIR}. Kept: {keep if keep else 'none'}")

Loaded pretrained DimeNet++ encoder (target=11)
Training TPNN | Target: 11 | Epochs: 50 | LR: 0.0001
Train: 193 | Val: 96 | Test: 194
Hierarchy/filtration visualization interval: every 2 epoch(s)
Gradient logging interval: every epoch (fixed)
------------------------------------------------------------
Epoch 000 | Train Loss: 384.884385 | Val Loss: 385.450151 | Val MAE: 19.466171
Epoch 001 | Train Loss: 116.326826 | Val Loss: 15.895495 | Val MAE: 3.361397
Epoch 002 | Train Loss: 3.520026 | Val Loss: 7.128009 | Val MAE: 2.163308
Epoch 003 | Train Loss: 2.424087 | Val Loss: 5.694442 | Val MAE: 1.958944
Epoch 004 | Train Loss: 1.512334 | Val Loss: 2.621394 | Val MAE: 1.305228
Epoch 005 | Train Loss: 1.260762 | Val Loss: 5.727568 | Val MAE: 2.027806
Epoch 006 | Train Loss: 1.212223 | Val Loss: 4.711813 | Val MAE: 1.872925
Epoch 007 | Train Loss: 1.073469 | Val Loss: 1.965490 | Val MAE: 1.159139
Epoch 008 | Train Loss: 0.779739 | Val Loss: 1.703330 | Val MAE: 1.064810
Epoch 009 | Train Loss